In [5]:
import arcpy
import os

arcpy.env.overwriteOutput = True

land_mask = r"C:\Users\acosta_pedro\OneDrive - Norges geologiske undersøkelse\Geochemistry NGU_2026\Kalk_project\Mask\Final_MDir\grunnkart_2025_fastlandsnorge__2_.shp"

snap_raster = r"E:\Alpha_earth\qc\alphaearth_mosaic_epsg25833_band1_qc_cog.tif"

out_cog = r"E:\mask\Mask_land_Kalk_cog.tif"

out_dir = os.path.dirname(out_cog)
os.makedirs(out_dir, exist_ok=True)

tmp_raster = os.path.join(out_dir, "tmp_Mask_land_Kalk.tif")

# Snap raster properties
desc = arcpy.Describe(snap_raster)
spatial_ref = desc.spatialReference

cell_size = float(
    arcpy.management.GetRasterProperties(
        snap_raster, "CELLSIZEX"
    ).getOutput(0)
)

# Environment: force same grid as snap raster
arcpy.env.snapRaster = snap_raster
arcpy.env.cellSize = snap_raster
arcpy.env.extent = snap_raster
arcpy.env.outputCoordinateSystem = spatial_ref
arcpy.env.mask = None

# Project polygon if needed
land_desc = arcpy.Describe(land_mask)

if land_desc.spatialReference.factoryCode != spatial_ref.factoryCode:
    projected_land = os.path.join(out_dir, "LandMask_projected.shp")

    if arcpy.Exists(projected_land):
        arcpy.management.Delete(projected_land)

    arcpy.management.Project(
        land_mask,
        projected_land,
        spatial_ref
    )

    land_to_rasterize = projected_land
else:
    land_to_rasterize = land_mask

# Add mask_val field if missing
fields = [f.name for f in arcpy.ListFields(land_to_rasterize)]

if "mask_val" not in fields:
    arcpy.management.AddField(
        land_to_rasterize,
        "mask_val",
        "SHORT"
    )

# Set all polygons to 1
arcpy.management.CalculateField(
    land_to_rasterize,
    "mask_val",
    1,
    "PYTHON3"
)

# Rasterize polygon: land = 1, outside = NoData
if arcpy.Exists(tmp_raster):
    arcpy.management.Delete(tmp_raster)

arcpy.conversion.PolygonToRaster(
    in_features=land_to_rasterize,
    value_field="mask_val",
    out_rasterdataset=tmp_raster,
    cell_assignment="CELL_CENTER",
    priority_field="NONE",
    cellsize=cell_size
)

# # ------------------------------------------------------------
# SAVE AS CLOUD OPTIMIZED GEOTIFF
# ------------------------------------------------------------
if arcpy.Exists(out_cog):
    arcpy.management.Delete(out_cog)

arcpy.management.CopyRaster(
    in_raster=tmp_raster,
    out_rasterdataset=out_cog,
    pixel_type="8_BIT_UNSIGNED",
    format="COG"
)

# Optional: build pyramids and statistics
# Some ArcGIS Pro versions do not support BuildPyramidsAndStatistics
try:
    arcpy.management.BuildPyramids(out_cog)
    arcpy.management.CalculateStatistics(out_cog)
except Exception as e:
    print("Warning: pyramids/statistics were not built.")
    print(e)

print("Done.")
print(f"Land mask saved as COG: {out_cog}")
print("Land cells = 1")
print("Outside polygon = NoData")

A raster error has occurred. The messages that follow will provide more detail.
Failed to build pyramid [Failed to build pyramid: Mask_land_Kalk_cog.tif: Cannot add external overviews when there are already internal overviews]
Failed to execute (BuildPyramids).

Done.
Land mask saved as COG: E:\mask\Mask_land_Kalk_cog.tif
Land cells = 1
Outside polygon = NoData


In [2]:
import arcpy

snap = r"E:\Alpha_earth\qc\alphaearth_mosaic_epsg25833_band1_qc_cog.tif"
mask = r"E:\mask\Mask_land_Kalk_cog.tif"

for r in [snap, mask]:
    print("\n", r)
    print("Rows:", arcpy.management.GetRasterProperties(r, "ROWCOUNT")[0])
    print("Cols:", arcpy.management.GetRasterProperties(r, "COLUMNCOUNT")[0])
    print("CellX:", arcpy.management.GetRasterProperties(r, "CELLSIZEX")[0])
    print("CellY:", arcpy.management.GetRasterProperties(r, "CELLSIZEY")[0])


 E:\Alpha_earth\qc\alphaearth_mosaic_epsg25833_band1_qc_cog.tif
Rows: 162232
Cols: 135236
CellX: 10
CellY: 10

 E:\mask\Mask_land_Kalk_cog.tif
Rows: 162232
Cols: 135236
CellX: 10
CellY: 10
